# 🛠️ Notebook 2 · Online Stock Brokerage — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/online-stock-brokerage
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


Here we turn the class design from Notebook 1 into **runnable code**: a tiny but functional brokerage.

We'll:
1. Build the domain classes.
2. Add order polymorphism (`MarketOrder`, `LimitOrder`, `StopOrder`).
3. Build a toy `Exchange` that matches orders when prices move.
4. Run end-to-end scenarios, including cancellation.


## 1️⃣ Domain: `Stock`, `Position`, `Portfolio`, `Account`

In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from itertools import count

@dataclass
class Stock:
    symbol: str
    price: float      # current market price - kept simple, one number

class Side(Enum):
    BUY = "buy"
    SELL = "sell"

class OrderStatus(Enum):
    PENDING   = "pending"
    FILLED    = "filled"
    CANCELLED = "cancelled"

@dataclass
class Position:
    symbol: str
    qty: int = 0
    avg_price: float = 0.0   # weighted-average cost basis

@dataclass
class Portfolio:
    positions: dict[str, Position] = field(default_factory=dict)

    def apply(self, symbol: str, qty_delta: int, price: float) -> None:
        """Apply a buy (+qty) or sell (-qty). Updates avg_price on buys."""
        pos = self.positions.setdefault(symbol, Position(symbol))
        if qty_delta > 0:  # buying - update weighted avg cost
            new_qty = pos.qty + qty_delta
            pos.avg_price = (pos.avg_price * pos.qty + price * qty_delta) / new_qty
            pos.qty = new_qty
        else:              # selling - avg_price unchanged
            pos.qty += qty_delta
        if pos.qty == 0:
            self.positions.pop(symbol, None)

@dataclass
class Account:
    id: int
    cash: float
    portfolio: Portfolio = field(default_factory=Portfolio)


## 2️⃣ Orders — one class per strategy

In [ ]:
_oids = count(1)   # simple id generator for orders

class Order(ABC):
    def __init__(self, account: Account, stock: Stock, side: Side, qty: int):
        if qty <= 0:
            raise ValueError("qty must be positive")
        self.id = next(_oids)
        self.account = account
        self.stock = stock
        self.side = side
        self.qty = qty
        self.status = OrderStatus.PENDING

    @abstractmethod
    def can_fill(self, market_price: float) -> bool: ...
    @abstractmethod
    def fill_price(self, market_price: float) -> float: ...

    def __repr__(self):
        extra = ""
        if isinstance(self, LimitOrder): extra = f" limit={self.limit_price}"
        if isinstance(self, StopOrder):  extra = f" stop={self.stop_price}"
        return f"#{self.id} {type(self).__name__}({self.side.value} {self.qty} {self.stock.symbol}{extra}, {self.status.value})"


class MarketOrder(Order):
    def can_fill(self, market_price): return True
    def fill_price(self, market_price): return market_price

class LimitOrder(Order):
    def __init__(self, account, stock, side, qty, limit_price):
        super().__init__(account, stock, side, qty)
        self.limit_price = limit_price
    def can_fill(self, market_price):
        return (market_price <= self.limit_price) if self.side == Side.BUY \
               else (market_price >= self.limit_price)
    def fill_price(self, market_price): return market_price

class StopOrder(Order):
    def __init__(self, account, stock, side, qty, stop_price):
        super().__init__(account, stock, side, qty)
        self.stop_price = stop_price
    def can_fill(self, market_price):
        return (market_price >= self.stop_price) if self.side == Side.BUY \
               else (market_price <= self.stop_price)
    def fill_price(self, market_price): return market_price


## 3️⃣ The `Exchange` — match orders, record trades

A real exchange maintains an **order book** with price-time priority (we'll build that in Notebook 3). For now we use a simpler model: an order sits in the book until a `tick` moves the price into its fill zone.


In [ ]:
@dataclass
class Trade:
    order_id: int
    symbol: str
    side: Side
    qty: int
    price: float
    ts: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

class Exchange:
    def __init__(self):
        self.book: list[Order] = []
        self.trades: list[Trade] = []
        self.stocks: dict[str, Stock] = {}   # the exchange owns the market data

    # --- public API ------------------------------------------------------
    def list_stock(self, stock: Stock) -> Stock:
        """Register a tradable symbol. The exchange -- not an order -- owns its price."""
        self.stocks[stock.symbol] = stock
        return stock

    def place(self, order: Order) -> Order:
        """Accept an order. Try to fill immediately; otherwise rest in the book."""
        self.list_stock(order.stock)          # auto-register on first use
        self.book.append(order)
        self._try_fill(order)
        return order

    def cancel(self, order_id: int) -> bool:
        """Cancel a resting (still pending) order by id."""
        for o in self.book:
            if o.id == order_id and o.status == OrderStatus.PENDING:
                o.status = OrderStatus.CANCELLED
                self.book.remove(o)
                return True
        return False

    def tick(self, symbol: str, new_price: float) -> None:
        """Market moved. Update the price FIRST, then try to fill resting orders.

        Earlier drafts of this method did `o.stock.price = new_price` *inside* the
        loop over the book. That looks equivalent, but it isn't: if no order happens
        to be resting on the symbol, the price silently never moves. Market data is
        the exchange's state, not a side effect of iterating someone's orders.
        """
        if symbol not in self.stocks:
            raise KeyError(f'{symbol} is not listed on this exchange')
        self.stocks[symbol].price = new_price
        for o in list(self.book):
            if o.stock.symbol == symbol and o.status == OrderStatus.PENDING:
                self._try_fill(o)

    # --- internal --------------------------------------------------------
    def _try_fill(self, o: Order) -> None:
        mp = o.stock.price
        if not o.can_fill(mp):
            return
        fp = o.fill_price(mp)
        cost = fp * o.qty

        if o.side == Side.BUY:
            if o.account.cash < cost:
                return  # not enough cash - stays pending
            o.account.cash -= cost
            o.account.portfolio.apply(o.stock.symbol, +o.qty, fp)
        else:  # SELL
            pos = o.account.portfolio.positions.get(o.stock.symbol)
            if not pos or pos.qty < o.qty:
                return  # no shorting in this toy - stays pending
            o.account.cash += cost
            o.account.portfolio.apply(o.stock.symbol, -o.qty, fp)

        o.status = OrderStatus.FILLED
        self.trades.append(Trade(o.id, o.stock.symbol, o.side, o.qty, fp))
        self.book.remove(o)


## 4️⃣ Scenario — Alice buys AAPL, then a limit fills on a dip

In [ ]:
ex = Exchange()
aapl = ex.list_stock(Stock("AAPL", 180.0))   # the exchange lists the symbol
alice = Account(id=1, cash=10_000)

# 1) Market buy: fills immediately at 180.
ex.place(MarketOrder(alice, aapl, Side.BUY, 10))

# 2) Limit buy below market: waits in the book.
lo = ex.place(LimitOrder(alice, aapl, Side.BUY, 5, limit_price=170))

print("After placing orders:")
print("  cash:", alice.cash, "| positions:", alice.portfolio.positions)
print("  book:", ex.book)

# 3) Price drops - the limit triggers.
ex.tick("AAPL", 168.0)

print("\nAfter price tick to 168:")
print("  cash:", alice.cash, "| positions:", alice.portfolio.positions)
print("  trades:")
for t in ex.trades: print("   ", t)


## 5️⃣ Scenario — cancel a resting order

In [ ]:
# Place a limit far from market, then cancel it before it can trigger.
far = ex.place(LimitOrder(alice, aapl, Side.BUY, 1, limit_price=50))
print("Before cancel, book:", ex.book)

ok = ex.cancel(far.id)
print("cancelled?", ok, "| status:", far.status, "| book:", ex.book)


## 6️⃣ Scenario -- stop-loss protects a profit

Alice bought AAPL at 180 and it's now at 195. She places a **stop-sell at 190** so that if
the price falls back through 190, she's automatically out with a small gain locked in.

Note that the rally to 195 is just another `ex.tick(...)` -- because the exchange owns the
market data, moving the price works whether or not anyone has an order resting.

In [ ]:
# The market rallies to 195. No resting orders on AAPL right now -- and that's fine,
# `tick` updates the exchange's own price, so nothing is lost.
ex.tick("AAPL", 195.0)
assert aapl.price == 195.0, 'tick must move the price even with an empty book'

stop = ex.place(StopOrder(alice, aapl, Side.SELL, 10, stop_price=190))
print("Stop armed:", stop, "| book:", ex.book)

ex.tick("AAPL", 189.0)                     # price falls through the stop
print("After fall to 189:")
print("  cash:", alice.cash)
print("  positions:", alice.portfolio.positions)
print("  last trade:", ex.trades[-1])

## 7️⃣ Recap

- One class per order type → no `if/elif` on `kind` anywhere in `Exchange`.
- `Exchange.place`, `cancel`, and `tick` make a tiny but complete API.
- The order lifecycle (`PENDING → FILLED / CANCELLED`) lives on the `Order` itself.

### Try it yourself

- Add a `TrailingStopOrder` that moves its stop as the price rises. You should only need to **add one class** — existing code should keep working.
- Add `cash_reserved` to avoid double-spending when multiple buy orders are resting.
- Print P&L per position (`(current_price - avg_price) * qty`).

➡️ Notebook 3 adds a real **order book with price-time priority**, an **Observer** for live quotes, and a **Decorator** for risk checks.


## 8️⃣ 🧪 Verify the implementation

The scenarios above *print* what happened. These assertions *prove* it -- including
the two rules the exchange must never break: cash and shares are conserved, and a
cancelled order never fills.

In [ ]:
def must_raise(exc, fn, *a, **kw):
    try:
        fn(*a, **kw)
    except exc:
        return True
    raise AssertionError(f'expected {exc.__name__}, nothing was raised')

def run_scenario():
    ex = Exchange()
    sym = ex.list_stock(Stock("TEST", 100.0))
    acct = Account(id=99, cash=1_000.0)
    return ex, sym, acct

# 1. A market buy fills immediately and moves exactly (price * qty) out of cash.
ex, s, a = run_scenario()
o = ex.place(MarketOrder(a, s, Side.BUY, 5))
assert o.status is OrderStatus.FILLED
assert a.cash == 500.0 and a.portfolio.positions['TEST'].qty == 5
assert len(ex.trades) == 1 and ex.trades[0].price == 100.0
assert o not in ex.book, 'a filled order must leave the book'

# 2. Weighted-average cost basis: 5 @ 100 then 5 @ 80 -> avg 90.
ex.tick("TEST", 80.0)
ex.place(MarketOrder(a, s, Side.BUY, 5))
assert a.portfolio.positions['TEST'].avg_price == 90.0

# 3. A limit buy above the market does NOT fill; it rests until the price comes to it.
ex, s, a = run_scenario()
lim = ex.place(LimitOrder(a, s, Side.BUY, 1, limit_price=90))
assert lim.status is OrderStatus.PENDING and lim in ex.book
ex.tick("TEST", 95.0); assert lim.status is OrderStatus.PENDING   # still too expensive
ex.tick("TEST", 89.0); assert lim.status is OrderStatus.FILLED    # now it is cheap enough
assert lim.fill_price(89.0) == 89.0

# 4. Cancel removes the order from the book and it can never fill afterwards.
ex, s, a = run_scenario()
doomed = ex.place(LimitOrder(a, s, Side.BUY, 1, limit_price=1))
assert ex.cancel(doomed.id) is True
assert doomed.status is OrderStatus.CANCELLED and doomed not in ex.book
assert ex.cancel(doomed.id) is False, 'cancelling twice must be a no-op, not a crash'
ex.tick("TEST", 0.5)
assert doomed.status is OrderStatus.CANCELLED, 'a cancelled order must never fill'
assert ex.trades == []

# 5. Conservation: you cannot spend cash you do not have, or sell shares you do not own.
ex, s, a = run_scenario()
broke = ex.place(MarketOrder(a, s, Side.BUY, 100))       # needs $10,000, has $1,000
assert broke.status is OrderStatus.PENDING and a.cash == 1_000.0
naked = ex.place(MarketOrder(a, s, Side.SELL, 1))        # owns nothing -> no shorting
assert naked.status is OrderStatus.PENDING and a.cash == 1_000.0

# 6. Round trip conserves value: buy 4 @ 100, sell 4 @ 120 -> +$80, flat position.
ex, s, a = run_scenario()
ex.place(MarketOrder(a, s, Side.BUY, 4))
ex.tick("TEST", 120.0)
ex.place(MarketOrder(a, s, Side.SELL, 4))
assert a.cash == 1_000.0 - 400.0 + 480.0 == 1_080.0
assert 'TEST' not in a.portfolio.positions, 'a flat position must be dropped, not kept at qty=0'

# 7. The exchange rejects ticks for symbols it never listed.
must_raise(KeyError, ex.tick, "NOPE", 1.0)

print('exchange invariants hold ✅')